# مسئلهٔ ۱ — آماده‌سازی split آموزش و اعتبارسنجی

این نوت‌بوک ۶۰۰ ویدئوی منتخب را در سطح **ویدئو** به دو بخش متوازن تقسیم می‌کند. تمام ۸ فریم یک ویدئو فقط در یک بخش قرار می‌گیرند تا نشت داده رخ ندهد.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_ROOT = Path(r'P:\NexarCollisionData')
FRAME_INDEX_PATH = DATA_ROOT / 'frame_index.csv'
RANDOM_SEED = 42
VALIDATION_SIZE = 0.20

In [2]:
frame_index = pd.read_csv(FRAME_INDEX_PATH)

assert len(frame_index) == 4_800, 'Expected 600 videos × 8 frames.'
frames_per_video = frame_index.groupby('video_id').size()
assert len(frames_per_video) == 600
assert (frames_per_video == 8).all(), 'Every video must have exactly 8 frames.'

video_table = (
    frame_index[['video_id', 'video_path', 'label']]
    .drop_duplicates('video_id')
    .reset_index(drop=True)
)
print(video_table['label'].value_counts().sort_index())
video_table.head()

label
0    300
1    300
Name: count, dtype: int64


,video_id,video_path,label
0,1094,P:\NexarCollisionData\train\negative\01094.mp4,0
1,343,P:\NexarCollisionData\train\positive\00343.mp4,1
2,444,P:\NexarCollisionData\train\positive\00444.mp4,1
3,1609,P:\NexarCollisionData\train\negative\01609.mp4,0
4,1717,P:\NexarCollisionData\train\negative\01717.mp4,0


In [3]:
train_videos, val_videos = train_test_split(
    video_table,
    test_size=VALIDATION_SIZE,
    stratify=video_table['label'],
    random_state=RANDOM_SEED,
)

train_videos = train_videos.assign(split='train')
val_videos = val_videos.assign(split='validation')
video_splits = pd.concat([train_videos, val_videos], ignore_index=True)

frame_splits = frame_index.merge(
    video_splits[['video_id', 'split']], on='video_id', how='inner', validate='many_to_one'
)

assert len(frame_splits) == len(frame_index)
assert frame_splits.groupby('video_id')['split'].nunique().eq(1).all()

video_splits.to_csv(DATA_ROOT / 'video_splits.csv', index=False)
frame_splits.to_csv(DATA_ROOT / 'frame_splits.csv', index=False)

print('Video counts:')
print(pd.crosstab(video_splits['split'], video_splits['label']))
print('\nFrame counts:')
print(pd.crosstab(frame_splits['split'], frame_splits['label']))

Video counts:
label         0    1
split               
train       240  240
validation   60   60

Frame counts:
label          0     1
split                 
train       1920  1920
validation   480   480


## خروجی

- `P:\NexarCollisionData\video_splits.csv`: یک ردیف به‌ازای هر ویدئو، برای آموزش مدل و ارزیابی در سطح ویدئو.
- `P:\NexarCollisionData\frame_splits.csv`: اطلاعات هر فریم همراه با split و برچسب آن.

در مرحلهٔ بعد از `video_splits.csv` برای آموزش baseline استفاده می‌کنیم.